In [ ]:
import pandas as pd
df = pd.read_csv('data/segments-icw-cj-mh-2020-08-20.csv', sep=',')
df.head()

In [ ]:
# df.columns
df = df.rename(
    columns={
        # MyHeritage (svenska bulk):
        'DNA-matchnings-ID': 'Match ID',
        'Namn': 'Name',
        'Matchnamn': 'Match',
        'Kromosom': 'Chr',
        'Starta plats': 'Start',
        'Avsluta plats': 'End',
        'Börja RSID': 'Start RSID',
        'Avsluta RSID': 'End RSID',
        'Centimorgan': 'cM',
        'Snp\'s': 'SNPs',
        # MyHeritage (Chromosome browser):
        'Match Name': 'Match',
        'Chromosome': 'Chr',
        'Start Location': 'Start',
        'End Location': 'End',
        'Centimorgans': 'cM',
        # FamilyTreeDNA:
        'Matching SNPs': 'SNPs'
    }
)
# We could also add these columns from DNA Painter:
# Confidence, Group, Side, Notes, Colour
df.head()

In [ ]:
df['Interval'] = df.apply(
    lambda row: pd.Interval(row['Start'], row['End'], closed='both'),
    axis='columns'
)
df.head()

In [ ]:
df[df['Chr'] == 22]

In [ ]:
matches = df[['Match ID', 'Name', 'Match']].drop_duplicates()
selected_matchids = matches['Match ID'][matches['Match'].str.contains('Match 10') | matches['Match'].str.contains('Match 11') | matches['Match'].str.contains('Match 12')]
selected_matchids.head()

In [ ]:
segments = df[['Match ID', 'Chr', 'Start', 'End', 'Start RSID', 'End RSID', 'cM', 'SNPs', 'Interval']]
# segments = segments[segments['Match ID'].isin(selected_matchids)]
segments.head()

In [ ]:
# https://en.wikipedia.org/wiki/Single-nucleotide_polymorphism
chr_bplen = {
    1: 214066000,
    2: 222889000,
    3: 186938000,
    4: 169035000,
    5: 170954000,
    6: 165022000,
    7: 149414000,
    8: 125148000,
    9: 107440000,
    10: 127894000,
    11: 129193000,
    12: 125198000,
    13: 93711000,
    14: 89344000,
    15: 73467000,
    16: 74037000,
    17: 73367000,
    18: 73078000,
    19: 56044000,
    20: 63317000,
    21: 33824000,
    22: 33786000
}

In [ ]:
heatmap = pd.DataFrame()
for chr, bplen in chr_bplen.items():
    segments_chr = segments[segments['Chr'] == chr]
    if segments_chr.empty:
        continue
    # Chr 1: 214066000 bp / 1000 samples ~ 200000 bp
    bp_samples = list(range(0, bplen, 200000))
    matching = [sum(
            segments_chr.apply(
                    lambda row: bp in row['Interval'],
                    axis='columns'
                )) for bp in bp_samples
        ]
    heatmap[chr] = pd.Series(matching,
            index = bp_samples)
    print(f'Appended {sum(matching)} matching samples for chromosome {chr}')

In [ ]:
heatmap.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
fig, ax = plt.subplots(figsize = (30, 12))
sns.heatmap(heatmap.transpose(), cmap ='RdYlGn')